# 04. Ventilation Raw (인공호흡기 원본 추출)

## 목적
코호트 환자의 인공호흡기 사용 데이터를 **1시간 단위**로 추출

## 입력
- `cohort_base.csv`: 기본 코호트

## 출력
- `ventilation_raw.csv`: 시간별 인공호흡기 상태 (1 row = 1 patient × 1 hour)

## 전처리 범위
- [x] 데이터 추출 (procedureevents)
- [x] 시간 범위 확장 (start~end → 1시간 단위)
- [ ] 슬라이딩 윈도우 적용 → merge 단계에서

## Item ID
- 225792: Invasive Mechanical Ventilation

In [ ]:
import duckdb
import pandas as pd
import os

DB_PATH = '../data/duckdb/mimic_total.duckdb'
INPUT_DIR = '../data/processed'
OUTPUT_DIR = '../data/processed'

con = duckdb.connect(DB_PATH)
print("=== 04. Ventilation Raw 추출 시작 ===")

## Step 1: 코호트 로드

In [ ]:
print("Step 1: 코호트 로드")

cohort_path = os.path.join(INPUT_DIR, 'cohort_base.csv')
df_cohort = pd.read_csv(cohort_path, parse_dates=['intime', 'outtime'])

print(f"✓ 코호트 로드 완료: {len(df_cohort):,}명")

con.register('cohort', df_cohort)

## Step 2: Ventilation 데이터 추출 (1시간 단위로 확장)

In [ ]:
print("\nStep 2: Ventilation 데이터 추출")

vent_query = """
WITH vent_expanded AS (
    -- procedureevents의 starttime~endtime을 1시간 단위로 확장
    SELECT
        c.stay_id,
        UNNEST(generate_series(
            date_trunc('hour', CAST(pe.starttime AS TIMESTAMP)),
            date_trunc('hour', CAST(pe.endtime AS TIMESTAMP)),
            INTERVAL '1' HOUR
        )) AS charttime_h
    FROM cohort c
    INNER JOIN procedureevents pe ON c.stay_id = pe.stay_id
    WHERE pe.itemid = '225792'  -- Invasive Mechanical Ventilation
      AND pe.starttime IS NOT NULL
      AND pe.endtime IS NOT NULL
)
SELECT
    stay_id,
    charttime_h,
    1 as vent_flag  -- 해당 시간에 ventilation 사용 중
FROM vent_expanded
GROUP BY stay_id, charttime_h
ORDER BY stay_id, charttime_h
"""

print("  쿼리 실행 중...")
df_vent = con.execute(vent_query).df()

print(f"✓ Ventilation 추출 완료")
print(f"  - 총 행 수: {len(df_vent):,}개 (ventilation 사용 중인 시점만)")
print(f"  - 고유 환자: {df_vent['stay_id'].nunique():,}명")

## Step 3: 통계 확인

In [ ]:
print("\nStep 3: 통계 확인")

# 환자당 ventilation 시간
vent_hours_per_patient = df_vent.groupby('stay_id').size()

print(f"\n=== Ventilation 통계 ===")
print(f"  - Ventilation 사용 환자: {df_vent['stay_id'].nunique():,}명")
print(f"  - 전체 코호트 대비: {df_vent['stay_id'].nunique() / len(df_cohort) * 100:.1f}%")
print(f"\n환자당 Ventilation 시간:")
print(f"  - 평균: {vent_hours_per_patient.mean():.1f}시간")
print(f"  - 중앙값: {vent_hours_per_patient.median():.1f}시간")
print(f"  - 최대: {vent_hours_per_patient.max():,}시간")

## Step 4: 저장

In [ ]:
print("\n" + "="*60)
print("CSV 저장")
print("="*60)

output_path = os.path.join(OUTPUT_DIR, 'ventilation_raw.csv')
df_vent.to_csv(output_path, index=False)

file_size = os.path.getsize(output_path) / (1024 * 1024)

print(f"\n✓ 저장 완료: ventilation_raw.csv")
print(f"  - 파일 크기: {file_size:.2f} MB")
print(f"  - 행 수: {len(df_vent):,}개")
print(f"  - 경로: {output_path}")
print(f"\n참고: vent_flag=1인 행만 포함 (0은 merge 시 생성)")

In [ ]:
print("\n=== 샘플 데이터 ===")
df_vent.head(10)

In [ ]:
con.close()
print("\n=== 04. Ventilation Raw 추출 완료 ===")